In [20]:
%reload_ext autoreload
%load_ext autoreload 
%autoreload 2
%reload_ext autoreload
# Reload the river.tree.hoeffding_tree_classifier module to get the updated methods
import importlib
import sys
from confluent_kafka import Producer
import pickle
import time
import socket

conf = {
    'bootstrap.servers': 'broker:9092',
    'group.id': 'a',
    'auto.offset.reset': 'latest'
}

producer = Producer(conf)

def delivery_report(err, msg):
    """Callback function to report the status of the message delivery."""
    if err is not None:
        print(f"Message delivery failed: {err}")
    else:
        print(f"Message delivered to {msg.topic()} [{msg.partition()}] at offset {msg.offset()}")

# Remove cached modules
if 'river.tree.hoeffding_tree_classifier' in sys.modules:
    del sys.modules['river.tree.hoeffding_tree_classifier']
if 'river.tree' in sys.modules:
    del sys.modules['river.tree']

# Reimport
from river import tree
from river.tree import HoeffdingTreeClassifier
import time

# Verify the method exists
if hasattr(HoeffdingTreeClassifier, '_find_parent_and_index'):
    print("✅ Modules reloaded - _find_parent_and_index method available")
else:
    print("⚠️  Warning: _find_parent_and_index method not found")

test = 0

def split_callback(split_info):  
    global test
    import time
    print("\n" + "="*80)
    print("🌳 SPLIT EVENT DETECTED!")
    print("="*80)
    print(f"iteration {test}")
    
    original_leaf = split_info['original_leaf']
    new_split_node = split_info['new_split_node']
    new_leaves = split_info['new_leaves']
    parent = split_info['parent']
    parent_branch = split_info['parent_branch']
    
    # Extract split node type and parameters
    split_node_type = type(new_split_node).__name__
    
    # Build branch parameters based on node type
    branch_params = {
        'feature': getattr(new_split_node, 'feature', None)
    }
    
    if hasattr(new_split_node, 'threshold'):
        branch_params['threshold'] = float(getattr(new_split_node, 'threshold'))
    
    if hasattr(new_split_node, 'value'):
        branch_params['value'] = getattr(new_split_node, 'value')
    
    if hasattr(new_split_node, 'radius'):
        branch_params['radius'] = float(getattr(new_split_node, 'radius'))
    
    if hasattr(new_split_node, '_mapping'):
        # For multiway splits, extract the mapping
        branch_params['feature_values'] = list(getattr(new_split_node, '_mapping', {}).keys())
    
    # Extract new leaf data
    new_leaves_data = []
    for idx, leaf in enumerate(new_leaves):
        leaf_data = {
            'node_id': getattr(leaf, 'node_id', None),
            'node_type': type(leaf).__name__,
            'depth': getattr(leaf, 'depth', 0),
            'branch_index': idx,
            'stats': {str(k): float(v) for k, v in getattr(leaf, 'stats', {}).items()},
            'total_weight': float(getattr(leaf, 'total_weight', 0)),
            'mc_correct_weight': float(getattr(leaf, '_mc_correct_weight', 0)),
            'nb_correct_weight': float(getattr(leaf, '_nb_correct_weight', 0)),
        }
        
        # Extract splitter data for each leaf
        if hasattr(leaf, 'splitters'):
            splitters_data = {}
            for feature_name, splitter in leaf.splitters.items():
                splitter_info = {
                    'type': type(splitter).__name__,
                    'feature_name': feature_name
                }
                
                # Check if Gaussian splitter
                if hasattr(splitter, '_att_dist_per_class'):
                    att_dist = splitter._att_dist_per_class
                    if att_dist:
                        first_val = next(iter(att_dist.values()))
                        
                        # Gaussian splitter
                        if hasattr(first_val, 'mu'):
                            gaussian_data = {}
                            distributions = {}
                            
                            for class_label, dist_obj in att_dist.items():
                                class_data = {
                                    'n_samples': float(dist_obj.n_samples) if hasattr(dist_obj, 'n_samples') else 0.0,
                                    'mu': float(dist_obj.mu) if hasattr(dist_obj, 'mu') else 0.0,
                                    'sigma': float(dist_obj.sigma) if hasattr(dist_obj, 'sigma') else 1.0
                                }
                                distributions[str(class_label)] = class_data
                            
                            gaussian_data['distributions'] = distributions
                            
                            if hasattr(splitter, '_min_per_class'):
                                gaussian_data['min_per_class'] = {
                                    str(k): float(v) for k, v in splitter._min_per_class.items()
                                }
                            
                            if hasattr(splitter, '_max_per_class'):
                                gaussian_data['max_per_class'] = {
                                    str(k): float(v) for k, v in splitter._max_per_class.items()
                                }
                            
                            splitter_info['gaussian_data'] = gaussian_data
                        
                        # Nominal splitter
                        elif isinstance(first_val, dict):
                            nominal_data = {
                                'class_distributions': {
                                    str(k): dict(v) for k, v in att_dist.items()
                                }
                            }
                            
                            if hasattr(splitter, '_att_values'):
                                nominal_data['unique_values'] = list(splitter._att_values)
                            
                            if hasattr(splitter, '_total_weight_observed'):
                                nominal_data['total_weight'] = float(splitter._total_weight_observed)
                            
                            splitter_info['nominal_data'] = nominal_data
                
                splitters_data[feature_name] = splitter_info
            
            leaf_data['splitters'] = splitters_data
        
        new_leaves_data.append(leaf_data)
    
    # Build complete split event payload
    split_event = {
        'update_type': 'split',
        'timestamp': __import__('time').time(),
        
        # Original leaf that was replaced
        'original_leaf_id': getattr(original_leaf, 'node_id', None),
        
        # New split node created
        'split_node': {
            'node_id': getattr(new_split_node, 'node_id', None),
            'node_type': split_node_type,
            'depth': getattr(new_split_node, 'depth', 0),
            'stats': {str(k): float(v) for k, v in getattr(new_split_node, 'stats', {}).items()},
            'branch_params': branch_params
        },
        
        # New leaf children
        'new_leaves': new_leaves_data,
        
        # Parent context
        'parent_context': {
            'parent_node_id': getattr(parent, 'node_id', None) if parent else None,
            'parent_branch': parent_branch
        }
    }
    
    print(f"✅ Split captured: {split_node_type} on feature '{branch_params.get('feature')}'")
    print(f"   Original leaf ID: {split_event['original_leaf_id']}")
    print(f"   New split node ID: {split_event['split_node']['node_id']}")
    print(f"   New leaves: {[leaf['node_id'] for leaf in new_leaves_data]}")
    print("="*80 + "\n")

    import os
    import time
    
    directory = 'pickle_data_lab/split_event'
    os.makedirs(directory, exist_ok=True)
    filename = f'pickle_data_lab/split_event/split_{int(time.time() * 10000)}.pickle'
    pickled_data = pickle.dumps(split_event, protocol=pickle.HIGHEST_PROTOCOL)
    try:
        producer.produce(
            topic="my_confluent_topic",
            key="testing".encode('utf-8'),           # <-- Pass the serialized key here
            value=pickled_data,
            callback=delivery_report
        )
        producer.poll(0)

    except Exception as e:
        print(f"Error producing message: {e}")

def leaf_update_callback(update_info):
    import os
    import time
        
    pickled_data = pickle.dumps(update_info, protocol=pickle.HIGHEST_PROTOCOL)
    try:
        producer.produce(
            topic="my_confluent_topic",
            key="testing".encode('utf-8'),           # <-- Pass the serialized key here
            value=pickled_data,
            callback=delivery_report
        )
        producer.flush(timeout=5)

    except Exception as e:
        print(f"Error producing message: {e}")
    

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
✅ Modules reloaded - _find_parent_and_index method available


%4|1761311496.902|CONFWARN|rdkafka#producer-5| [thrd:app]: Configuration property group.id is a consumer property and will be ignored by this producer instance
%4|1761311496.902|CONFWARN|rdkafka#producer-5| [thrd:app]: Configuration property auto.offset.reset is a consumer property and will be ignored by this producer instance


In [26]:
update_tree = HoeffdingTreeClassifier(
            grace_period=200,
            leaf_prediction='nba',
            leaf_update_threshold=1,
            leaf_update_callback=leaf_update_callback,
            split_callback=split_callback
        )
from river.datasets import synth
dataset = synth.Agrawal(classification_function=0, seed=42)

# import shutil
# directory = 'pickle_data_lab'
# shutil.rmtree(directory, ignore_errors=True)
for i, (x, y) in enumerate(dataset.take(5000)): #2819
    # print(f"iteration {i}")
    test = i
    update_tree.learn_one(x, y)

   🏷️  REGISTERED NODE: ID=0, Type=LeafNaiveBayesAdaptive

🚪 GATE 2 TRIGGERED: Leaf 0 reached 1 instances
   Previous weight: 0.0 → Current weight: 1.0
   Threshold: 1 (multiple #1)
Message delivered to my_confluent_topic [1] at offset 18952
   ✅ GATE 2 CALLBACK: Executed successfully

🚪 GATE 2 TRIGGERED: Leaf 0 reached 2 instances
   Previous weight: 1.0 → Current weight: 2.0
   Threshold: 1 (multiple #2)
Message delivered to my_confluent_topic [1] at offset 18953
   ✅ GATE 2 CALLBACK: Executed successfully

🚪 GATE 2 TRIGGERED: Leaf 0 reached 3 instances
   Previous weight: 2.0 → Current weight: 3.0
   Threshold: 1 (multiple #3)
Message delivered to my_confluent_topic [1] at offset 18954
   ✅ GATE 2 CALLBACK: Executed successfully

🚪 GATE 2 TRIGGERED: Leaf 0 reached 4 instances
   Previous weight: 3.0 → Current weight: 4.0
   Threshold: 1 (multiple #4)
Message delivered to my_confluent_topic [1] at offset 18955
   ✅ GATE 2 CALLBACK: Executed successfully

🚪 GATE 2 TRIGGERED: Leaf 0 re

In [27]:
# inference with update process
results = []
dataset = synth.Agrawal(classification_function=0, seed=44)
for i, (x, y) in enumerate(dataset.take(100)):
    results.append(update_tree.predict_proba_one(x))

In [29]:
results

[{0: 0.0, 1: 1.0},
 {0: 0.0, 1: 1.0},
 {0: 0.0, 1: 1.0},
 {0: 0.0, 1: 1.0},
 {0: 1.0, 1: 0.0},
 {0: 0.0, 1: 1.0},
 {0: 1.0, 1: 0.0},
 {0: 1.0, 1: 0.0},
 {0: 0.0, 1: 1.0},
 {0: 0.0, 1: 1.0},
 {0: 0.0, 1: 1.0},
 {0: 1.0, 1: 0.0},
 {0: 0.0, 1: 1.0},
 {0: 1.0, 1: 0.0},
 {0: 1.0, 1: 0.0},
 {0: 0.0, 1: 1.0},
 {0: 0.0, 1: 1.0},
 {0: 0.0, 1: 1.0},
 {0: 1.0, 1: 0.0},
 {0: 0.0, 1: 1.0},
 {0: 1.0, 1: 0.0},
 {0: 0.0, 1: 1.0},
 {0: 0.0, 1: 1.0},
 {0: 1.0, 1: 0.0},
 {0: 0.0, 1: 1.0},
 {0: 0.0, 1: 1.0},
 {0: 1.0, 1: 0.0},
 {0: 1.0, 1: 0.0},
 {0: 0.0, 1: 1.0},
 {0: 0.0, 1: 1.0},
 {0: 0.0, 1: 1.0},
 {0: 0.0, 1: 1.0},
 {0: 0.0, 1: 1.0},
 {0: 1.0, 1: 0.0},
 {0: 0.0, 1: 1.0},
 {0: 1.0, 1: 0.0},
 {0: 0.0, 1: 1.0},
 {0: 0.0, 1: 1.0},
 {0: 0.0, 1: 1.0},
 {0: 1.0, 1: 0.0},
 {0: 0.0, 1: 1.0},
 {0: 1.0, 1: 0.0},
 {0: 1.0, 1: 0.0},
 {0: 1.0, 1: 0.0},
 {0: 0.0, 1: 1.0},
 {0: 0.0, 1: 1.0},
 {0: 0.0, 1: 1.0},
 {0: 0.0, 1: 1.0},
 {0: 1.0, 1: 0.0},
 {0: 0.0, 1: 1.0},
 {0: 0.0, 1: 1.0},
 {0: 0.0, 1: 1.0},
 {0: 0.0, 1:

In [28]:
# read from json_data_lab/test_results.json and compare it with results
with open('json_data_lab/test_results.json', 'r') as f:
    import json
    json_results = json.load(f)

for i in range(len(results)):
    r1 = results[i]
    r2_raw = json_results[i]
    # convert string keys to integer keys and ensure float values
    r2 = {int(k): float(v) for k, v in r2_raw.items()}
    for key in r1.keys():
        v1 = r1[key]
        v2 = r2.get(key, 0.0)

        assert abs(v1 - v2) < 1e-6, f"Mismatch at index {i} for class {key}: {v1} vs {v2}"
print("✅ Inference results match the expected values.")

✅ Inference results match the expected values.
